# DEXTRA · 01 — Download external data (ERA5 + FIRMS → Hugging Face)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aduitya9/DEXTRA/blob/main/notebooks/01_download_external.ipynb)

Downloads everything the DEXTRA pipeline needs **except CPCB station data**, and stores it in your private Hugging Face dataset repo:

| Section | Source | What | Typical time |
|---|---|---|---|
| A | Open-Meteo (ERA5) | Hourly surface weather at the 15 cities, 2019–2025 | ~10 min |
| A2 | Copernicus CDS (ERA5) | Boundary-layer height for Jan–Jun 2024, which Open-Meteo is missing | minutes to an hour |
| B | Copernicus CDS (ERA5) | Temperature at 850–950 hPa + 2 m temperature at 00/03 UTC, for the inversion index | minutes to hours (CDS queue) |
| C | NASA FIRMS | MODIS active-fire detections over India, 2019–2025 | ~10–20 min |

CPCB station data can't be scripted (the CPCB repository is behind a CAPTCHA), so it is downloaded in a browser separately and uploaded to the same repo.

### Before you run: add these 4 Colab Secrets
Click the **🔑 key icon** in the left sidebar → **Add new secret**, and switch **Notebook access** on for each:

| Secret name | Value |
|---|---|
| `HF_TOKEN` | Hugging Face token with **write** access to your dataset repo |
| `HF_DATASET_REPO` | Your dataset repo id, e.g. `your-username/dextra-aqi-india` |
| `CDSAPI_KEY` | Your Copernicus CDS API key (CDS profile page → *API key*) |
| `FIRMS_MAP_KEY` | Your NASA FIRMS MAP_KEY |

Never paste keys into a cell: anything in a cell is saved with the notebook and can end up on GitHub.

### Runtime
This notebook only downloads, so it doesn't need a GPU. It runs fine on a **T4** runtime, but a **CPU** runtime (*Runtime → Change runtime type → CPU*) saves your free GPU hours for training.

### Safe to re-run
Every finished piece is uploaded to Hugging Face immediately, and every section skips what is already there. If Colab disconnects, just **Runtime → Run all** again.


## 0 · Settings

In [ ]:
# Leave HF_DATASET_REPO empty to read it from the HF_DATASET_REPO Colab secret.
HF_DATASET_REPO = ""
GITHUB_REPO_URL = "https://github.com/Aduitya9/DEXTRA.git"

RUN_METADATA = True          # city table + Diwali dates
RUN_ERA5_SURFACE = True      # Section A
RUN_BLH_GAP_FILL = True      # Section A2
RUN_ERA5_INVERSION = True    # Section B
RUN_FIRMS = True             # Section C

# Lower to 6, 3 or 1 if CDS rejects a request as too large.
CDS_MONTHS_PER_REQUEST = 12

## 1 · Secrets
Read from Colab Secrets. Values are never printed — only whether each one was found.

In [ ]:
from google.colab import userdata

def colab_secret(name, required=True):
    """Read a Colab secret without ever printing its value."""
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab secret '{name}' is missing or this notebook can't read it. Open the key icon "
            "(Secrets) in the left sidebar, add it, and switch on 'Notebook access'."
        )
    return value

HF_TOKEN = colab_secret("HF_TOKEN")
HF_DATASET_REPO = HF_DATASET_REPO or colab_secret("HF_DATASET_REPO")
CDSAPI_KEY = colab_secret("CDSAPI_KEY", required=RUN_ERA5_INVERSION or RUN_BLH_GAP_FILL)
FIRMS_MAP_KEY = colab_secret("FIRMS_MAP_KEY", required=RUN_FIRMS)

print("Dataset repo:", HF_DATASET_REPO)
print("Secrets found:", {"HF_TOKEN": bool(HF_TOKEN), "CDSAPI_KEY": bool(CDSAPI_KEY), "FIRMS_MAP_KEY": bool(FIRMS_MAP_KEY)})

## 2 · Install packages and get the DEXTRA code

In [ ]:
%pip install -q "cdsapi>=0.7.7" netCDF4 "huggingface_hub>=0.23.0"

In [ ]:
import os
import subprocess
import sys

REPO_DIR = "/content/DEXTRA"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", GITHUB_REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import pandas as pd
from src import config
from src.data import calendar_events, download_era5, download_firms
from src.utils.hf_store import HFStore

store = HFStore(HF_DATASET_REPO, token=HF_TOKEN)
YEARS = list(range(int(config.DATA_START_DATE[:4]), int(config.DATA_END_DATE[:4]) + 1))
CITY_SLUG = {city: city.lower() for city in config.TARGET_CITIES}

print(f"Connected to {HF_DATASET_REPO}: {len(store.files())} files already there")
print(f"Years: {YEARS[0]}–{YEARS[-1]}, cities: {len(config.TARGET_CITIES)}")

## 3 · Metadata
The city table and the Diwali dates the feature pipeline uses. Small, so it is re-uploaded every run (unchanged files make no new commit).

In [ ]:
if RUN_METADATA:
    cities = pd.DataFrame([{"city": city, **meta} for city, meta in config.TARGET_CITIES.items()])
    for path, df in {
        "meta/cities.csv": cities,
        "meta/diwali_dates.csv": calendar_events.diwali_table(),
    }.items():
        store.upload_df(df, path, f"Metadata: {path}")
        print(f"↑ {path}: {len(df)} rows")

## A · ERA5 surface weather (Open-Meteo)
Hourly boundary-layer height, 2 m temperature and dewpoint, surface pressure, precipitation and 10 m wind (m/s) at the ERA5 grid cell nearest each city, in UTC. One Parquet file per city: `raw/era5_surface/<city>.parquet`.

In [ ]:
if RUN_ERA5_SURFACE:
    for city in config.TARGET_CITIES:
        path = f"raw/era5_surface/{CITY_SLUG[city]}.parquet"
        if store.exists(path):
            print(f"✓ {city}: already on Hugging Face, skipped")
            continue
        df = download_era5.fetch_surface_city(city, YEARS)
        store.upload_df(df, path, f"ERA5 surface (Open-Meteo) for {city}, {YEARS[0]}-{YEARS[-1]}")
        other_missing = df.drop(columns=["city", "time_utc", "blh", "grid_lat", "grid_lon"]).isna().mean().max()
        print(f"↑ {city}: {len(df):,} hourly rows, grid cell ({df.grid_lat.iloc[0]}, {df.grid_lon.iloc[0]}), "
              f"BLH missing {df.blh.isna().mean():.1%} (patched in A2), other variables missing ≤ {other_missing:.1%}")

## A2 · Boundary-layer height gap fill (Copernicus CDS)
Open-Meteo's ERA5 archive is missing boundary-layer height for **every hour of Jan–Jun 2024**, in every city, which is exactly DEXTRA's validation period. This section finds the missing months in the files from Section A, fetches them from CDS, and also fetches one neighbouring month that both sources have, then prints how closely they agree. Both are the same ERA5 reanalysis, so the difference should be near zero.

The patch is stored separately (`raw/era5_blh_patch/<year>.parquet`) so the raw Open-Meteo files stay untouched; `02_build_dataset` merges them.

In [ ]:
if RUN_BLH_GAP_FILL:
    import cdsapi

    cds = cdsapi.Client(url=config.CDS_URL, key=CDSAPI_KEY)
    surface = pd.concat(
        [store.read_df(f"raw/era5_surface/{CITY_SLUG[city]}.parquet") for city in config.TARGET_CITIES],
        ignore_index=True,
    )
    gaps = download_era5.blh_gap_months(surface)
    print("Months with missing boundary-layer height:", gaps or "none")

    for year, months in gaps.items():
        path = f"raw/era5_blh_patch/{year}.parquet"
        if store.exists(path):
            print(f"✓ {year}: patch already on Hugging Face, skipped")
            continue
        check_month = download_era5.overlap_month(months)
        print(f"… {year}: requesting months {months} + overlap month {check_month} from CDS")
        patch = download_era5.download_blh_months(cds, year, months + [check_month], "/content/era5_tmp", CDS_MONTHS_PER_REQUEST)

        agreement = download_era5.blh_agreement(surface, patch)
        display(agreement.round(3))
        if len(agreement) and agreement["correlation"].min() < 0.95:
            print("⚠ CDS and Open-Meteo disagree more than expected; check the agreement table before using the patch.")
        store.upload_df(patch, path, f"ERA5 boundary-layer height patch (CDS) {year}, months {months}")
        print(f"↑ {year}: {len(patch):,} hourly rows")

## B · ERA5 inversion inputs (Copernicus CDS)
Temperature at 850, 875, 900, 925 and 950 hPa, plus 2 m temperature and surface pressure, at 00 and 03 UTC (05:30 and 08:30 IST, when inversions are strongest). Each city later uses the level ~75 hPa above its own surface, because a fixed 925 hPa is underground in Bengaluru.

**This section can be slow:** CDS queues requests, and a year's request can wait from minutes to hours. If Colab disconnects, run again; finished years are skipped. One file per year: `raw/era5_inversion/<year>.parquet`.

In [ ]:
if RUN_ERA5_INVERSION:
    import cdsapi

    cds = cdsapi.Client(url=config.CDS_URL, key=CDSAPI_KEY)
    for year in YEARS:
        path = f"raw/era5_inversion/{year}.parquet"
        if store.exists(path):
            print(f"✓ {year}: already on Hugging Face, skipped")
            continue
        print(f"… {year}: submitting CDS requests (they may wait in the CDS queue)")
        df = download_era5.download_inversion_year(cds, year, "/content/era5_tmp", CDS_MONTHS_PER_REQUEST)
        store.upload_df(df, path, f"ERA5 inversion inputs (CDS) {year}")
        print(f"↑ {year}: {len(df):,} rows, {df.city.nunique()} cities, columns: {[c for c in df.columns if c.startswith('temp')]}")

## C · NASA FIRMS fire detections
MODIS Collection 6.1 detections over India and its neighbours (the Pakistani Punjab and Bangladesh lie inside the 500 km transport radius). Quality-controlled standard-processing data is used wherever it exists; the last few months, which it doesn't cover yet, come from near-real-time data and are marked in the `source` column. One file per year: `raw/firms/modis_<year>.parquet`.

In [ ]:
if RUN_FIRMS:
    try:
        availability = download_firms.data_availability(FIRMS_MAP_KEY)
        sp_end = download_firms.archive_end_date(availability, download_firms.SP_SOURCE)
    except RuntimeError as err:
        print("Could not read FIRMS data availability, using standard processing throughout:", err)
        sp_end = None
    print("MODIS standard-processing archive ends:", sp_end)

    for year in YEARS:
        path = f"raw/firms/modis_{year}.parquet"
        if store.exists(path):
            print(f"✓ {year}: already on Hugging Face, skipped")
            continue
        df = download_firms.fetch_year(FIRMS_MAP_KEY, year, sp_end)
        store.upload_df(df, path, f"NASA FIRMS MODIS fire detections {year}")
        sources = df["source"].value_counts().to_dict() if len(df) else {}
        print(f"↑ {year}: {len(df):,} detections {sources}")

## D · Check what is on Hugging Face
Reads every raw file back and compares row counts with what each source should produce.

In [ ]:
store.refresh()
expected_hours = len(pd.date_range(f"{YEARS[0]}-01-01", f"{YEARS[-1]}-12-31 23:00", freq="h"))

rows = []
for path in store.files("raw/"):
    df = store.read_df(path)
    if path.startswith("raw/era5_surface/"):
        expected = expected_hours
    elif path.startswith("raw/era5_inversion/"):
        year = int(path.rsplit("/", 1)[1][:4])
        expected = len(config.TARGET_CITIES) * len(pd.date_range(f"{year}-01-01", f"{year}-12-31")) * len(config.INVERSION_HOURS_UTC)
    else:
        expected = None
    ok = (len(df) == expected) if expected else (len(df) > 0)
    rows.append({"file": path, "rows": len(df), "expected": expected or "> 0", "ok": "✓" if ok else "✗"})

summary = pd.DataFrame(rows)
display(summary)
print(f"{(summary.ok == '✓').sum()} of {len(summary)} files look complete")

## Next
1. **CPCB station data:** downloaded in a browser (CAPTCHA), then uploaded to `raw/cpcb/` in the same repo.
2. **`02_build_dataset.ipynb`:** hourly → daily with CPCB's rules, city medians, AQI, merge with ERA5 + FIRMS, train/val/test splits.